In [ ]:
import numpy as np
import pandas as pd

SEED_DIGIT = 6                         
SEED = SEED_DIGIT
N_BOOT = 200 + 10 * SEED_DIGIT


X = pd.read_csv("secom.data", sep=" ", header=None)
X.columns = [f"f{i}" for i in X.columns]
lab = pd.read_csv("secom_labels.data", sep=" ", header=None,
                  names=["label", "timestamp"], quotechar='"')
outcome = lab["label"].map({-1: "Pass", 1: "Fail"}).to_numpy()
is_pass, is_fail = outcome == "Pass", outcome == "Fail"


ok = [c for c in X.columns
      if X.loc[is_pass, c].notna().sum() >= 10
      and X.loc[is_fail, c].notna().sum() >= 10
      and X[c].nunique(dropna=True) > 1]
feats = sorted(np.random.default_rng(SEED).choice(ok, 5, replace=False).tolist())

S = X[feats].to_numpy(dtype=float)
S_imp = np.where(np.isnan(S), np.nanmedian(S, axis=0), S)   
P_av, F_av = S[is_pass], S[is_fail]
P_im, F_im = S_imp[is_pass], S_imp[is_fail]


obs_av = np.nanmean(F_av, axis=0) - np.nanmean(P_av, axis=0)
obs_im = F_im.mean(axis=0) - P_im.mean(axis=0)


def bootstrap(seed):
    rng = np.random.default_rng(seed)
    nP, nF = len(P_av), len(F_av)
    d_av = np.empty((N_BOOT, len(feats)))
    d_im = np.empty((N_BOOT, len(feats)))
    for b in range(N_BOOT):
        ip = rng.integers(0, nP, nP)      
        jf = rng.integers(0, nF, nF)    
        d_av[b] = np.nanmean(F_av[jf], axis=0) - np.nanmean(P_av[ip], axis=0)
        d_im[b] = F_im[jf].mean(axis=0) - P_im[ip].mean(axis=0)
    return d_av, d_im

d_av, d_im = bootstrap(SEED)

def summarize(d):
    return d.mean(axis=0), np.percentile(d, 2.5, axis=0), np.percentile(d, 97.5, axis=0)

bm_av, lo_av, hi_av = summarize(d_av)
bm_im, lo_im, hi_im = summarize(d_im)


rank_av = pd.Series(np.abs(obs_av), index=feats).rank(ascending=False).astype(int)
rank_im = pd.Series(np.abs(obs_im), index=feats).rank(ascending=False).astype(int)
sign_changed = np.sign(obs_av) != np.sign(obs_im)
rank_changed = (rank_av != rank_im).to_numpy()
ci_av_sig = (lo_av > 0) | (hi_av < 0)     
ci_im_sig = (lo_im > 0) | (hi_im < 0)
change = np.abs(obs_im - obs_av)

comp = pd.DataFrame({
    "feature": feats,
    "obs_diff_available": obs_av, "boot_mean_available": bm_av,
    "ci_low_available": lo_av, "ci_high_available": hi_av, "rank_available": rank_av.values,
    "obs_diff_imputed": obs_im, "boot_mean_imputed": bm_im,
    "ci_low_imputed": lo_im, "ci_high_imputed": hi_im, "rank_imputed": rank_im.values,
    "abs_change_in_diff": change,
    "sign_changed": sign_changed, "rank_changed": rank_changed,
    "ci_excludes_zero_available": ci_av_sig, "ci_excludes_zero_imputed": ci_im_sig,
})
comp.to_csv("robustness_comparison.csv", index=False)


d_av2, d_im2 = bootstrap(SEED)
repro = np.array_equal(d_av, d_av2) and np.array_equal(d_im, d_im2)

top = comp.loc[comp["abs_change_in_diff"].idxmax()]
changed_sign = [f for f, c in zip(feats, sign_changed) if c]
changed_rank = [f for f, c in zip(feats, rank_changed) if c]

report = f"""SENSITIVITY REPORT
Selected features: {feats}
Bootstrap repetitions: {N_BOOT} (stratified, seed {SEED})
Difference = mean(Fail) - mean(Pass)

Largest change in mean difference: {top['feature']} ({top['abs_change_in_diff']:.6f})
Features with changed sign: {changed_sign or 'None'}
Features with changed ranking: {changed_rank or 'None'}
CI excludes zero (available-case): {[f for f, c in zip(feats, ci_av_sig) if c] or 'None'}
CI excludes zero (median-imputed): {[f for f, c in zip(feats, ci_im_sig) if c] or 'None'}
Reproducibility verified: {'Yes' if repro else 'No'}

Interpretation: Median imputation fills missing values with the center of the
data, which pulls group means toward the overall median and usually shrinks the
Fail-Pass difference relative to available-case analysis. Features whose sign
or ranking stay the same under both strategies, and whose 95% CI excludes zero
in both, are the most robust. Features whose CI spans zero, or that change sign
or rank, should not be used for engineering decisions without more data.
"""
open("sensitivity_report.txt", "w").write(report)

print("Selected Features:", feats)
print("Bootstrap Repetitions:", N_BOOT)
print("Largest Change in Mean Difference:", top["feature"], round(top["abs_change_in_diff"], 6))
print("Features with Changed Difference Sign:", changed_sign or "None")
print("Features with Changed Ranking:", changed_rank or "None")
print("Reproducibility Verified:", "Yes" if repro else "No")

Selected Features: ['f171', 'f228', 'f289', 'f300', 'f564']
Bootstrap Repetitions: 260
Largest Change in Mean Difference: f564 0.116403
Features with Changed Difference Sign: None
Features with Changed Ranking: ['f300', 'f564']
Reproducibility Verified: Yes
